# Figure 9

In [ ]:

from pathlib import Path
import csv
import json
import re

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator, MultipleLocator

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans", "Liberation Sans"]
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.size"] = 12
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["legend.frameon"] = False


def find_project_root():
    candidate = Path.cwd().resolve()
    while candidate != candidate.parent:
        if (candidate / "Hannah_Inversion_GPT").is_dir():
            return candidate
        candidate = candidate.parent
    raise FileNotFoundError("Could not find project root containing Hannah_Inversion_GPT")


ROOT = find_project_root()
DATA = ROOT / "logdata" / "Hannah"
MODEL = ROOT / "Hannah_Inversion_GPT"
FIGURE_DIR = ROOT / "Figure" / "Well_validation"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE = ROOT / ".backup_hannah_before_five_well_cleanup_20260807"


def read_csv(name):
    with (DATA / name).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def read_csv_file(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


wells = {row["api"]: row for row in read_csv("hannah_named_well_model_comparison.csv")}
evidence_by_api = {}
for row in read_csv("hannah_named_well_model_evidence_intervals.csv"):
    evidence_by_api.setdefault(row["api"], []).append(row)

# The archived altered-ophiolitic proxy well is retained when its optional
# backup directory is present.  The active Hannah data remain usable even
# after that historical archive has been removed.
if ARCHIVE.is_dir():
    PROXY_APIS = {"03390278"}
    archive_wells = read_csv_file(ARCHIVE / "hannah_lithology_model_match.csv")
    for row in archive_wells:
        if row["api"] in PROXY_APIS:
            wells[row["api"]] = row

    archive_evidence = read_csv_file(ARCHIVE / "hannah_pdf_model_evidence_intervals.csv")
    for row in archive_evidence:
        if row["api"] not in PROXY_APIS:
            continue
        # The broad TD/formation statement for 03390278 is not a continuous
        # lithology log; retain only its depth-specific altered-greenstone interval.
        if row["api"] == "03390278" and "altered greenstone" not in row["observed"].lower():
            continue
        row["evidence_quality"] = "B"
        evidence_by_api.setdefault(row["api"], []).append(row)

# The archived proxy record is no longer stored as active well data, but its
# previously verified location and depth-specific PDF interval are retained
# here so the historical PNG can be regenerated reproducibly.
if "03390278" not in wells:
    wells["03390278"] = {
        "api": "03390278",
        "well_name": "NE Geysers Unit 8",
        "x_m": "520750.0",
        "y_m": "4298750.0",
        "topography_m": "812.820",
        "depth_m": "2962.960",
        "bottom_elevation_m": "-2150.140",
        "model_cell_ijk": "[21, 17, 11]",
    }
    evidence_by_api["03390278"] = [{
        "top_m": "1336.728",
        "bottom_m": "1577.400",
        "observed": "highly altered greenstone rubble, then chert/greenstone",
        "basis": "reported drilling interval approximately 1336.7-1577.4 m",
        "evidence_quality": "B",
    }]


def parse_mesh():
    lines = (MODEL / "mesh" / "mesh_core.msh").read_text(encoding="utf-8").splitlines()
    nx, ny, nz = (int(value) for value in lines[0].split())
    origin = np.asarray([float(value) for value in lines[1].split()], dtype=float)
    widths = np.asarray([float(value) for line in lines[2:] for value in line.split()], dtype=float)
    if widths.size != nx + ny + nz:
        raise ValueError(f"Unexpected mesh width count: {widths.size}")
    wx = widths[:nx]
    wy = widths[nx:nx + ny]
    wz = widths[nx + ny:]
    x_edges = origin[0] + np.r_[0.0, np.cumsum(wx)]
    y_edges = origin[1] + np.r_[0.0, np.cumsum(wy)]
    # geo_id_3d uses k=0 at the deepest cell, so reverse UBC z widths.
    z_edges = origin[2] - wz.sum() + np.r_[0.0, np.cumsum(wz[::-1])]
    return x_edges, y_edges, z_edges


x_edges, y_edges, z_edges = parse_mesh()
unit = np.load(MODEL / "geology_models" / "unit_id_3d.npy")
topography_xyz = np.loadtxt(MODEL / "topo" / "topography.xyz", usecols=(0, 1, 2))

# Hand-picked categorical colors: keep units distinct while giving the set a muted cool-warm tone.
UNIT_COLORS = [
    "#D9D9D9", "#6BAED6", "#9ECAE1", "#7B9ACC", "#A6A6C8",
    "#F2C6B4", "#E98B6A", "#B6A0C9", "#8C78B8", "#C95A49",
]
UNIT_LABELS = {
    1: "Regional background",
    2: "Geysers plutonic complex / GVS",
    3: "GVS & melt body",
    4: "Greenstone / ophiolite mafic rocks",
    5: "Coast Range / GVS ophiolite",
    6: "Lower-degree serpentinite",
    7: "Ophiolite mélange / altered serpentinite",
    8: "Mafic volcanic stock or plug",
    9: "Magnetic volcanic stock or plug",
    10: "Serpentinite main target",
}
UNIT_CMAP = ListedColormap(UNIT_COLORS)
UNIT_NORM = BoundaryNorm(np.arange(0.5, 10.5 + 1.0, 1.0), UNIT_CMAP.N)

LITH_COLORS = {
    "serpentine-bearing / altered": "#4D9221",
    "greenstone / volcanic": "#35978F",
    "graywacke / argillite": "#8C8C8C",
    "Franciscan formation": "#C7C7C7",
    "depth only / lithology insufficient": "#D9C2A6",
}


def compact_label(text, max_chars=34):
    text = re.sub(r"\s+", " ", text.strip())
    return text if len(text) <= max_chars else text[:max_chars - 1] + "..."


def lithology_class(observed):
    text = observed.lower()
    if "serpentine" in text:
        return "serpentine-bearing / altered"
    if "greenstone" in text or "volcanic" in text:
        return "greenstone / volcanic"
    if "graywacke" in text or "argillite" in text:
        return "graywacke / argillite"
    if "franciscan" in text:
        return "Franciscan formation"
    return "depth only / lithology insufficient"


def evidence_hatch(quality):
    quality = quality.upper()
    if quality.startswith("A"):
        return None
    if quality.startswith("B"):
        return "//"
    return ".."


def evidence_is_depth_specific(evidence):
    text = " ".join(
        str(evidence.get(key, "")) for key in ("observed", "basis", "evidence_quality")
    ).lower()
    return "no continuous" not in text and "broad formation" not in text


plot_evidence_by_api = {
    api: [evidence for evidence in rows if evidence_is_depth_specific(evidence)]
    for api, rows in evidence_by_api.items()
}


def nearest_topography_line(axis_values, fixed_value, orientation="xz"):
    xy = topography_xyz[:, :2]
    line = []
    for axis_value in axis_values:
        if orientation == "xz":
            distance2 = (xy[:, 0] - axis_value) ** 2 + (xy[:, 1] - fixed_value) ** 2
        elif orientation == "yz":
            distance2 = (xy[:, 0] - fixed_value) ** 2 + (xy[:, 1] - axis_value) ** 2
        else:
            raise ValueError(f"Unsupported section orientation: {orientation}")
        line.append(topography_xyz[int(np.argmin(distance2)), 2])
    return np.asarray(line)


def model_cell_from_row(row):
    if row.get("model_cell_ijk"):
        return tuple(int(value) for value in json.loads(row["model_cell_ijk"]))
    return tuple(int(float(row[key])) for key in ("i", "j", "k_bottom"))


LITH_CODES = {
    "serpentine-bearing / altered": "S",
    "greenstone / volcanic": "G",
    "graywacke / argillite": "W",
    "Franciscan formation": "F",
    "depth only / lithology insufficient": "?",
}


def cell_evidence(depth_top, depth_bottom, evidences):
    best = None
    best_overlap = 0.0
    for evidence in evidences:
        overlap = max(
            0.0,
            min(depth_bottom, float(evidence["bottom_m"]))
            - max(depth_top, float(evidence["top_m"])),
        )
        if overlap > best_overlap:
            best = evidence
            best_overlap = overlap
    return best


def plot_well_section(api, z_limits=None, evidence_only=False):
    row = wells[api]
    i, j, _ = model_cell_from_row(row)
    x_well = float(row["x_m"])
    y_well = float(row["y_m"])
    topo = float(row["topography_m"])
    depth = float(row["depth_m"])
    bottom = float(row["bottom_elevation_m"])
    section = unit[:, j, :].T
    x_line = np.linspace(x_edges[0], x_edges[-1], 160)
    topo_line = nearest_topography_line(x_line, y_well)
    # Use one common vertical window for the regular sections.  A custom
    # window is supported for focused views such as the Kettenhofen shallow
    # elevation range.
    if z_limits is None:
        z_low = max(z_edges[0], -4000.0)
        z_high = min(z_edges[-1], topo + 500.0)
    else:
        z_low, z_high = z_limits

    fig, ax = plt.subplots(figsize=(9.0, 6.8))
    ax.pcolormesh(x_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.plot(x_line, topo_line, color="#222222", linewidth=1.0, label="topography")
    ax.fill_between(x_line, topo_line, z_high, color="white", alpha=0.70, zorder=3)

    # Overlay a column of the exact model cells crossed by the well.  The
    # column uses the model x/z resolution, not an arbitrary line width.
    x0, x1 = x_edges[i], x_edges[i + 1]
    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        overlap = max(0.0, min(cell_high, topo) - max(cell_low, bottom))
        if overlap <= 0.0:
            continue
        z0 = max(cell_low, bottom)
        z1 = min(cell_high, topo)
        depth_top = max(0.0, topo - z1)
        depth_bottom = min(depth, topo - z0)
        evidence_rows = (
            plot_evidence_by_api.get(api, [])
            if evidence_only
            else evidence_by_api.get(api, [])
        )
        evidence = cell_evidence(depth_top, depth_bottom, evidence_rows)
        if evidence is None:
            if evidence_only:
                continue
            facecolor = "none"
            edgecolor = "#333333"
            hatch = None
            alpha = 0.85
            label = ""
        else:
            category = lithology_class(evidence["observed"])
            facecolor = LITH_COLORS[category]
            edgecolor = "white"
            hatch = evidence_hatch(evidence.get("evidence_quality", "C"))
            alpha = 0.88
            label = LITH_CODES[category]
        ax.add_patch(
            Rectangle(
                (x0, z0), x1 - x0, z1 - z0,
                facecolor=facecolor, edgecolor=edgecolor, linewidth=1.0,
                hatch=hatch, alpha=alpha, zorder=6,
            )
        )
        if label and (z1 - z0) >= 250.0:
            ax.text(
                (x0 + x1) / 2.0, (z0 + z1) / 2.0, label,
                ha="center", va="center", fontsize=8, fontweight="bold",
                color="#111111", zorder=7,
            )
    if not evidence_only:
        ax.add_patch(Rectangle((x0, bottom), x1 - x0, topo - bottom, fill=False, edgecolor="#111111", linewidth=1.4, zorder=7))
        ax.scatter([x_well], [topo], s=28, color="#111111", zorder=8)
        ax.scatter([x_well], [bottom], s=26, facecolor="white", edgecolor="#111111", zorder=8)
        if z_low <= topo + 120.0 <= z_high:
            ax.text((x0 + x1) / 2.0, topo + 120.0, "well grid", ha="center", va="bottom", fontsize=7, zorder=8)

    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel("UTM Easting (m)")
    ax.set_ylabel("Elevation (m)")
    title_suffix = "  |  PDF depth-specific lithology only" if evidence_only else ""
    ax.set_title(f"{row['well_name']}  |  API {api}{title_suffix}", loc="left", fontsize=10, fontweight="bold")
    secondary = ax.secondary_yaxis(
        "right",
        functions=(lambda elevation: topo - elevation, lambda below_surface: topo - below_surface),
    )
    secondary.set_ylabel("Depth below surface (m)")

    unit_handles = [Patch(facecolor=UNIT_COLORS[idx - 1], label=f"U{idx}: {UNIT_LABELS[idx]}") for idx in UNIT_LABELS]
    lith_handles = [Patch(facecolor=color, label=f"{LITH_CODES[label]}: {label}") for label, color in LITH_COLORS.items()]
    lith_handles.append(Patch(facecolor="white", edgecolor="#333333", label="—: no PDF evidence"))
    fig.subplots_adjust(left=0.09, right=0.93, top=0.91, bottom=0.33)
    fig.legend(handles=unit_handles, loc="lower left", bbox_to_anchor=(0.09, 0.16), ncol=5, fontsize=6.2, handlelength=1.2, columnspacing=1.0)
    fig.legend(handles=lith_handles, loc="lower left", bbox_to_anchor=(0.09, 0.07), ncol=3, fontsize=6.4, handlelength=1.2, columnspacing=1.0)
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", row["well_name"]).strip("_")
    output_base = FIGURE_DIR / f"{api}_{safe_name}_xz"
    fig.savefig(f"{output_base}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output_base


def plot_named_well_elevation_summary(
    api, z_limits, evidence_top_elevation, evidence_bottom_elevation, evidence_unit,
):
    """Plot a focused elevation section with a compact, gridless well track."""
    row = wells[api]
    i, j, _ = model_cell_from_row(row)
    x_well = float(row["x_m"])
    z_low, z_high = z_limits
    sample_top = evidence_top_elevation
    sample_bottom = evidence_bottom_elevation
    section = unit[:, j, :].T

    fig = plt.figure(figsize=(11.2, 6.8))
    grid = fig.add_gridspec(1, 2, width_ratios=[18.0, 1.55], wspace=0.18)
    ax = fig.add_subplot(grid[0, 0])
    track = fig.add_subplot(grid[0, 1], sharey=ax)

    ax.pcolormesh(x_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.scatter([x_well], [z_high], s=42, color="#111111", zorder=8, clip_on=False)
    ax.text(
        x_well, z_high - 70.0, "well position",
        ha="center", va="top", fontsize=7, color="#111111", zorder=8,
    )
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel("UTM Easting (m)")
    ax.set_ylabel("Elevation (m)")
    ax.set_title(
        f"{row['well_name']}  |  API {api}  |  Elevation {z_high:.0f} to {z_low:.0f} m",
        loc="left", fontsize=10, fontweight="bold",
    )

    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        if min(cell_high, z_high) <= max(cell_low, z_low):
            continue
        z0 = max(cell_low, z_low)
        z1 = min(cell_high, z_high)
        sample_overlap = max(
            0.0,
            min(cell_high, sample_top) - max(cell_low, sample_bottom),
        )
        assigned_unit = evidence_unit if sample_overlap > 0.0 else 1
        track.add_patch(
            Rectangle(
                (0.20, z0), 0.60, z1 - z0,
                facecolor=UNIT_COLORS[assigned_unit - 1],
                edgecolor="none", linewidth=0.0, zorder=3,
            )
        )
    evidence_label_low = max(z_low, sample_bottom)
    evidence_label_high = min(z_high, sample_top)
    if evidence_label_low < evidence_label_high:
        track.text(
            0.50, (evidence_label_low + evidence_label_high) / 2.0,
            f"U{evidence_unit}",
            ha="center", va="center", fontsize=8, fontweight="bold",
            color="#111111", zorder=4,
        )
    track.add_patch(
        Rectangle(
            (0.20, z_low), 0.60, z_high - z_low,
            facecolor="none", edgecolor="#222222", linewidth=0.8, zorder=5,
        )
    )
    track.set_xlim(0.0, 1.0)
    track.set_ylim(z_low, z_high)
    track.set_xticks([])
    track.tick_params(axis="y", left=False, labelleft=False)
    track.set_title("Well column", fontsize=8, pad=6)
    track.spines["left"].set_visible(False)
    track.spines["right"].set_visible(False)
    track.spines["top"].set_visible(False)

    unit_handles = [
        Patch(facecolor=UNIT_COLORS[idx - 1], label=f"U{idx}: {UNIT_LABELS[idx]}")
        for idx in UNIT_LABELS
    ]
    fig.subplots_adjust(left=0.08, right=0.96, top=0.90, bottom=0.25)
    fig.legend(
        handles=unit_handles, loc="lower center", bbox_to_anchor=(0.50, 0.025),
        ncol=5, fontsize=6.2, handlelength=1.2, columnspacing=1.0,
    )
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", row["well_name"]).strip("_")
    output_base = FIGURE_DIR / f"{api}_{safe_name}_xz"
    fig.savefig(f"{output_base}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output_base


def plot_neasham_elevation_summary(api):
    """Plot Neasham from 0 to -2300 m elevation with a separate well track."""
    return plot_named_well_elevation_summary(
        api,
        z_limits=(-2300.0, 0.0),
        evidence_top_elevation=-1511.0,
        evidence_bottom_elevation=-1512.5,
        evidence_unit=5,
    )


def plot_kettenhofen_elevation_summary(api):
    """Plot Kettenhofen from 0 to -2000 m elevation with the S interval as U6."""
    row = wells[api]
    topo = float(row["topography_m"])
    evidence_top_depth = 737.616
    evidence_bottom_depth = 1166.294
    return plot_named_well_elevation_summary(
        api,
        z_limits=(-2000.0, 0.0),
        evidence_top_elevation=topo - evidence_top_depth,
        evidence_bottom_elevation=topo - evidence_bottom_depth,
        evidence_unit=6,
    )


def plot_geyser_proxy_xz_summary(api):
    """Plot NE Geysers Unit 8 from 0 to -2000 m elevation with the altered-greenstone interval as U4."""
    row = wells[api]
    topo = float(row["topography_m"])
    evidence_top_depth = 1336.728
    evidence_bottom_depth = 1577.400
    return plot_named_well_elevation_summary(
        api,
        z_limits=(-2000.0, 0.0),
        evidence_top_elevation=topo - evidence_top_depth,
        evidence_bottom_elevation=topo - evidence_bottom_depth,
        evidence_unit=4,
    )


def plot_dx_state_elevation_summary(api):
    """Plot DX State 4596-30 from +1000 to -2000 m elevation as model U1."""
    return plot_named_well_elevation_summary(
        api,
        z_limits=(-2000.0, 1000.0),
        evidence_top_elevation=1000.0,
        evidence_bottom_elevation=-2000.0,
        evidence_unit=1,
    )


def draw_section_panel(ax, api, orientation, show_depth_axis=False, z_limits=None):
    """Draw one mesh-aligned XZ or YZ panel with the well evidence column."""
    row = wells[api]
    i, j, _ = model_cell_from_row(row)
    topo = float(row["topography_m"])
    depth = float(row["depth_m"])
    bottom = float(row["bottom_elevation_m"])
    if orientation == "xz":
        axis_edges = x_edges
        section = unit[:, j, :].T
        axis_well = float(row["x_m"])
        cell0, cell1 = x_edges[i], x_edges[i + 1]
        fixed_value = float(row["y_m"])
        axis_label = "UTM Easting (m)"
    elif orientation == "yz":
        axis_edges = y_edges
        section = unit[i, :, :].T
        axis_well = float(row["y_m"])
        cell0, cell1 = y_edges[j], y_edges[j + 1]
        fixed_value = float(row["x_m"])
        axis_label = "UTM Northing (m)"
    else:
        raise ValueError(f"Unsupported section orientation: {orientation}")

    axis_line = np.linspace(axis_edges[0], axis_edges[-1], 160)
    topo_line = nearest_topography_line(axis_line, fixed_value, orientation)
    if z_limits is None:
        z_low = max(z_edges[0], -4000.0)
        z_high = min(z_edges[-1], topo + 500.0)
    else:
        z_low, z_high = z_limits
    ax.pcolormesh(axis_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.plot(axis_line, topo_line, color="#222222", linewidth=1.0)
    ax.fill_between(axis_line, topo_line, z_high, color="white", alpha=0.70, zorder=3)

    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        overlap = max(0.0, min(cell_high, topo) - max(cell_low, bottom))
        if overlap <= 0.0:
            continue
        z0 = max(cell_low, bottom)
        z1 = min(cell_high, topo)
        depth_top = max(0.0, topo - z1)
        depth_bottom = min(depth, topo - z0)
        evidence = cell_evidence(depth_top, depth_bottom, evidence_by_api[api])
        if evidence is None:
            facecolor, edgecolor, hatch, alpha, label = "none", "#333333", None, 0.85, ""
        else:
            category = lithology_class(evidence["observed"])
            facecolor = LITH_COLORS[category]
            edgecolor = "white"
            hatch = evidence_hatch(evidence.get("evidence_quality", "C"))
            alpha = 0.88
            label = LITH_CODES[category]
        ax.add_patch(
            Rectangle(
                (cell0, z0), cell1 - cell0, z1 - z0,
                facecolor=facecolor, edgecolor=edgecolor, linewidth=1.0,
                hatch=hatch, alpha=alpha, zorder=6,
            )
        )
        if label and (z1 - z0) >= 250.0:
            ax.text(
                (cell0 + cell1) / 2.0, (z0 + z1) / 2.0, label,
                ha="center", va="center", fontsize=8, fontweight="bold",
                color="#111111", zorder=7,
            )
    ax.add_patch(
        Rectangle(
            (cell0, bottom), cell1 - cell0, topo - bottom,
            fill=False, edgecolor="#111111", linewidth=1.4, zorder=7,
        )
    )
    ax.scatter([axis_well], [topo], s=28, color="#111111", zorder=8)
    ax.scatter([axis_well], [bottom], s=26, facecolor="white", edgecolor="#111111", zorder=8)
    if z_low <= topo + 120.0 <= z_high:
        ax.text((cell0 + cell1) / 2.0, topo + 120.0, "well grid", ha="center", va="bottom", fontsize=7, zorder=8)
    ax.set_xlim(axis_edges[0], axis_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel(axis_label)
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)
    ax.set_ylabel("Elevation (m)")
    if show_depth_axis:
        secondary = ax.secondary_yaxis(
            "right",
            functions=(lambda elevation: topo - elevation, lambda below_surface: topo - below_surface),
        )
        secondary.set_ylabel("Depth below surface (m)")


def plot_wilson_zy_summary(api):
    """Plot the Wilson Y-Z section from -1000 to -3000 m elevation."""
    row = wells[api]
    i, j, _ = model_cell_from_row(row)
    y_well = float(row["y_m"])
    topo = float(row["topography_m"])
    bottom = float(row["bottom_elevation_m"])
    z_low, z_high = -3000.0, -1000.0
    section = unit[i, :, :].T

    fig = plt.figure(figsize=(11.2, 6.8))
    grid = fig.add_gridspec(1, 2, width_ratios=[18.0, 1.55], wspace=0.18)
    ax = fig.add_subplot(grid[0, 0])
    track = fig.add_subplot(grid[0, 1], sharey=ax)

    ax.pcolormesh(y_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.scatter([y_well], [z_high], s=42, color="#111111", zorder=8, clip_on=False)
    ax.text(
        y_well, z_high - 70.0, "well position",
        ha="center", va="top", fontsize=7, color="#111111", zorder=8,
    )
    ax.set_xlim(y_edges[0], y_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel("UTM Northing (m)")
    ax.set_ylabel("Elevation (m)")
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)
    ax.set_title(
        f"{row['well_name']}  |  API {api}  |  Y-Z section  |  Elevation -1000 to -3000 m",
        loc="left", fontsize=10, fontweight="bold",
    )

    secondary = ax.secondary_yaxis(
        "right",
        functions=(lambda elevation: topo - elevation, lambda below_surface: topo - below_surface),
    )
    secondary.set_ylabel("Depth below surface (m)")

    evidence_rows = plot_evidence_by_api.get(api, [])
    visible_evidences = []
    for evidence in evidence_rows:
        evidence_top = topo - float(evidence["top_m"])
        evidence_bottom = topo - float(evidence["bottom_m"])
        visible_low = max(z_low, evidence_bottom)
        visible_high = min(z_high, evidence_top)
        if visible_low < visible_high:
            visible_evidences.append((evidence, visible_low, visible_high))

    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        z0 = max(cell_low, z_low, bottom)
        z1 = min(cell_high, z_high, topo)
        if z1 <= z0:
            continue
        depth_top = max(0.0, topo - z1)
        depth_bottom = min(float(row["depth_m"]), topo - z0)
        evidence = cell_evidence(depth_top, depth_bottom, evidence_rows)
        if evidence is None:
            facecolor = UNIT_COLORS[0]
        else:
            facecolor = LITH_COLORS[lithology_class(evidence["observed"])]
        track.add_patch(
            Rectangle(
                (0.20, z0), 0.60, z1 - z0,
                facecolor=facecolor, edgecolor="none", linewidth=0.0, zorder=3,
            )
        )

    for evidence, visible_low, visible_high in visible_evidences:
        category = lithology_class(evidence["observed"])
        track.text(
            0.50, (visible_low + visible_high) / 2.0, LITH_CODES[category],
            ha="center", va="center", fontsize=8, fontweight="bold",
            color="#111111", zorder=4,
        )
    track.add_patch(
        Rectangle(
            (0.20, z_low), 0.60, z_high - z_low,
            facecolor="none", edgecolor="#222222", linewidth=0.8, zorder=5,
        )
    )
    track.set_xlim(0.0, 1.0)
    track.set_ylim(z_low, z_high)
    track.set_xticks([])
    track.tick_params(axis="y", left=False, labelleft=False)
    track.set_title("PDF lithology", fontsize=8, pad=6)
    track.spines["left"].set_visible(False)
    track.spines["right"].set_visible(False)
    track.spines["top"].set_visible(False)

    unit_handles = [Patch(facecolor=UNIT_COLORS[idx - 1], label=f"U{idx}: {UNIT_LABELS[idx]}") for idx in UNIT_LABELS]
    lith_handles = [Patch(facecolor=color, label=f"{LITH_CODES[label]}: {label}") for label, color in LITH_COLORS.items()]
    fig.subplots_adjust(left=0.08, right=0.96, top=0.90, bottom=0.27)
    fig.legend(
        handles=unit_handles, loc="lower center", bbox_to_anchor=(0.50, 0.085),
        ncol=5, fontsize=6.2, handlelength=1.2, columnspacing=1.0,
    )
    fig.legend(
        handles=lith_handles, loc="lower center", bbox_to_anchor=(0.50, 0.025),
        ncol=3, fontsize=6.4, handlelength=1.2, columnspacing=1.0,
    )
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", row["well_name"]).strip("_")
    output_base = FIGURE_DIR / f"{api}_{safe_name}_zy"
    fig.savefig(f"{output_base}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output_base


def draw_five_panel_elevation(ax, track, api, panel_title, z_limits, evidence_top_elevation, evidence_bottom_elevation, evidence_unit, show_xaxis=True, show_track_title=True):
    """Draw one X-Z panel directly on a shared multi-panel Matplotlib figure."""
    row = wells[api]
    _, j, _ = model_cell_from_row(row)
    x_well = float(row["x_m"])
    topo = float(row["topography_m"])
    z_low, z_high = z_limits
    section = unit[:, j, :].T

    ax.pcolormesh(x_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.scatter([x_well], [z_high], s=28, color="#111111", zorder=8, clip_on=False)
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel("Easting (m)", fontsize=17.2, labelpad=3)
    ax.set_ylabel("Elevation (m)", fontsize=17.2, labelpad=3)
    ax.tick_params(axis="both", labelsize=13.2, pad=2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MultipleLocator(1000))
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)
    ax.set_title(panel_title, loc="left", fontsize=17.0, fontweight="bold", pad=3)
    secondary = ax.secondary_yaxis(
        "right", functions=(lambda elevation: topo - elevation, lambda depth: topo - depth),
    )
    secondary.set_ylabel("Depth below surface (m)", fontsize=14.0, labelpad=3)
    secondary.tick_params(labelsize=13.0, pad=2)
    secondary.yaxis.set_major_locator(MultipleLocator(1000))
    if not show_xaxis:
        ax.set_xlabel("")
        ax.tick_params(axis="x", which="both", bottom=True, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(True)

    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        if min(cell_high, z_high) <= max(cell_low, z_low):
            continue
        z0 = max(cell_low, z_low)
        z1 = min(cell_high, z_high)
        sample_overlap = max(
            0.0,
            min(cell_high, evidence_top_elevation) - max(cell_low, evidence_bottom_elevation),
        )
        assigned_unit = evidence_unit if sample_overlap > 0.0 else 1
        track.add_patch(
            Rectangle(
                (0.20, z0), 0.60, z1 - z0,
                facecolor=UNIT_COLORS[assigned_unit - 1], edgecolor="none", linewidth=0.0,
            )
        )
    evidence_label_low = max(z_low, evidence_bottom_elevation)
    evidence_label_high = min(z_high, evidence_top_elevation)
    if evidence_label_low < evidence_label_high:
        track.text(
            0.50, (evidence_label_low + evidence_label_high) / 2.0, f"U{evidence_unit}",
            ha="center", va="center", fontsize=14.5, fontweight="bold", color="#111111",
        )
    track.add_patch(
        Rectangle((0.20, z_low), 0.60, z_high - z_low, facecolor="none", edgecolor="#222222", linewidth=0.7)
    )
    track.set_xlim(0.0, 1.0)
    track.set_ylim(z_low, z_high)
    track.set_xticks([])
    track.tick_params(axis="y", left=False, labelleft=False)
    if show_track_title:
        track.set_title("well", fontsize=17.5, pad=5)
    track.spines["left"].set_visible(False)
    track.spines["right"].set_visible(False)
    track.spines["top"].set_visible(False)


def draw_five_panel_wilson(ax, track, api, panel_title, show_xaxis=True, show_track_title=True):
    """Draw the Wilson Y-Z panel directly on the shared multi-panel figure."""
    row = wells[api]
    i, _, _ = model_cell_from_row(row)
    y_well = float(row["y_m"])
    topo = float(row["topography_m"])
    bottom = float(row["bottom_elevation_m"])
    z_low, z_high = -3000.0, -1000.0
    section = unit[i, :, :].T

    ax.pcolormesh(y_edges, z_edges, section, cmap=UNIT_CMAP, norm=UNIT_NORM, shading="flat")
    ax.scatter([y_well], [z_high], s=28, color="#111111", zorder=8, clip_on=False)
    ax.set_xlim(y_edges[0], y_edges[-1])
    ax.set_ylim(z_low, z_high)
    ax.set_xlabel("Northing (m)", fontsize=17.2, labelpad=3)
    ax.set_ylabel("Elevation (m)", fontsize=17.2, labelpad=3)
    ax.tick_params(axis="both", labelsize=13.2, pad=2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MultipleLocator(1000))
    ax.ticklabel_format(axis="x", style="plain", useOffset=False)
    ax.set_title(panel_title, loc="left", fontsize=17.0, fontweight="bold", pad=3)
    secondary = ax.secondary_yaxis(
        "right", functions=(lambda elevation: topo - elevation, lambda depth: topo - depth),
    )
    secondary.set_ylabel("Depth below surface (m)", fontsize=14.0, labelpad=3)
    secondary.tick_params(labelsize=13.0, pad=2)
    secondary.yaxis.set_major_locator(MultipleLocator(1000))
    if not show_xaxis:
        ax.set_xlabel("")
        ax.tick_params(axis="x", which="both", bottom=True, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(True)

    evidence_rows = plot_evidence_by_api.get(api, [])
    for k in range(len(z_edges) - 1):
        cell_low = float(z_edges[k])
        cell_high = float(z_edges[k + 1])
        z0 = max(cell_low, z_low, bottom)
        z1 = min(cell_high, z_high, topo)
        if z1 <= z0:
            continue
        depth_top = max(0.0, topo - z1)
        depth_bottom = min(float(row["depth_m"]), topo - z0)
        evidence = cell_evidence(depth_top, depth_bottom, evidence_rows)
        facecolor = UNIT_COLORS[0] if evidence is None else LITH_COLORS[lithology_class(evidence["observed"])]
        track.add_patch(Rectangle((0.20, z0), 0.60, z1 - z0, facecolor=facecolor, edgecolor="none", linewidth=0.0))

    visible_intervals = []
    for evidence in evidence_rows:
        evidence_top = topo - float(evidence["top_m"])
        evidence_bottom = topo - float(evidence["bottom_m"])
        visible_low = max(z_low, evidence_bottom)
        visible_high = min(z_high, evidence_top)
        if visible_low < visible_high:
            visible_intervals.append((visible_low, visible_high))
    if visible_intervals:
        label_low = min(interval[0] for interval in visible_intervals)
        label_high = max(interval[1] for interval in visible_intervals)
        track.text(
            0.50, (label_low + label_high) / 2.0, "Mixed",
            ha="center", va="center", fontsize=14.5, fontweight="bold", color="#111111",
        )
    track.add_patch(Rectangle((0.20, z_low), 0.60, z_high - z_low, facecolor="none", edgecolor="#222222", linewidth=0.7))
    track.set_xlim(0.0, 1.0)
    track.set_ylim(z_low, z_high)
    track.set_xticks([])
    track.tick_params(axis="y", left=False, labelleft=False)
    if show_track_title:
        track.set_title("well", fontsize=17.5, pad=5)
    track.spines["left"].set_visible(False)
    track.spines["right"].set_visible(False)
    track.spines["top"].set_visible(False)


def compose_five_well_sections():
    """Create the five-row figure from the model arrays, with shared legends."""
    fig = plt.figure(figsize=(15.5, 16.0))
    # A narrow spacer separates the final Wilson panel from the four X-Z panels.
    grid = fig.add_gridspec(
        6, 2, width_ratios=[18.0, 1.55], height_ratios=[1.0, 1.0, 1.0, 1.0, 0.01125, 1.0], wspace=0.18, hspace=0.20,
    )
    panel_axes = []

    configurations = [
        ("09790201", "(a) DX State 4596-30", (-2000.0, 1000.0), 1000.0, -2000.0, 1),
        ("03390278", "(b) NE Geysers Unit 8", (-2000.0, 0.0), 812.820 - 1336.728, 812.820 - 1577.400, 4),
        ("03390013", "(c) Kettenhofen No. 1 (K1)", (-2000.0, 0.0), 588.229 - 737.616, 588.229 - 1166.294, 6),
        ("03390254", "(d) Neasham No. 1 (N1)", (-2300.0, 0.0), -1511.0, -1512.5, 5),
    ]
    for row_index, (api, panel_title, z_limits, evidence_top, evidence_bottom, evidence_unit) in enumerate(configurations):
        ax = fig.add_subplot(grid[row_index, 0])
        track = fig.add_subplot(grid[row_index, 1], sharey=ax)
        draw_five_panel_elevation(ax, track, api, panel_title, z_limits, evidence_top, evidence_bottom, evidence_unit, show_xaxis=row_index >= 3, show_track_title=row_index == 0)
        panel_axes.append(ax)

    ax = fig.add_subplot(grid[5, 0])
    track = fig.add_subplot(grid[5, 1], sharey=ax)
    draw_five_panel_wilson(ax, track, "03390256", "(e) Wilson No. 1", show_xaxis=True, show_track_title=False)
    panel_axes.append(ax)

    unit_handles = [
        Patch(facecolor=UNIT_COLORS[idx - 1], label=f"U{idx}: {UNIT_LABELS[idx]}")
        for idx in UNIT_LABELS
    ]
    fig.subplots_adjust(left=0.10, right=0.95, top=0.985, bottom=0.185)
    for handles, y, ncol in (
        (unit_handles[:3], 0.105, 3),
        (unit_handles[3:6], 0.065, 3),
        (unit_handles[6:], 0.025, 4),
    ):
        fig.legend(
            handles=handles, loc="lower center", bbox_to_anchor=(0.50, y),
            ncol=ncol, fontsize=14.0, handlelength=1.3, columnspacing=1.0, frameon=False,
        )
    # Keep the shared legend at its existing regular weight; bold all other figure text.
    legend_text_ids = {id(text) for legend in fig.legends for text in legend.get_texts()}
    for text in fig.findobj(matplotlib.text.Text):
        if id(text) not in legend_text_ids:
            text.set_fontweight("bold")

    output = ROOT / "Figure" / "Figure9_Hannah_well_validation.png"
    fig.savefig(output, dpi=600, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return output


five_well_composite = compose_five_well_sections()
print("Generated well validation figure:")
print(five_well_composite)
